In [55]:
import pandas as pd
import networkx as nx

def build_network_by_index(
    B_nodes_csv,
    AA_edges_csv,
    AB_edges_csv,
    has_header=True
):
    header = 0 if has_header else None
    G = nx.Graph()

    # Add Nodes
    nodes = pd.read_csv(B_nodes_csv, header=header)
    for i, row in nodes.iterrows():
        G.add_node(row.iloc[0], direction=row.iloc[1])

    # Add Eeges
    for edge_csv in [AA_edges_csv, AB_edges_csv]:
        edges = pd.read_csv(edge_csv, header=header)
        for i, row in edges.iterrows():
            G.add_edge(row.iloc[0],row.iloc[1],direction=row.iloc[2])

    # Define layer 0
    layer_0 = []
    for n in nodes.iloc[:, 0]:
        if G.degree(n) > 0:
            layer_0.append(n)
    
    return G,layer_0


In [127]:
def build_layers(G, layer_0, max_depth=None):
    # list of sets --> layers[d] will contain all nodes at distance d (starting will change later)
    layers = []

    
    layers.append(set(layer_0))    # "layer 0" is the source nodes
    visited = set(layer_0)    # "visited" is all nodes we have already visited
    current_layer = set(layer_0)    # nodes at the current distance from layer 0
    depth = 0    # current distance from layer 0

    
    while current_layer: #until no new nodes are found
        if max_depth is not None and depth >= max_depth:    # until max depth (DO NOT REMOVE KEPT FOR DEBUGGING)
            break

        next_layer = set()  # next_layer nodes = distance (depth + 1)

        for node in current_layer:    # for every node in the current layer
            for neighbor in G.neighbors(node):     # look at all neighbors of this node
                if neighbor not in visited:     # add neighbor only if it hasn't appeared in any earlier layer
                    next_layer.add(neighbor)

        if not next_layer:     # if no new nodes were found, we are done
            break

        layers.append(next_layer)  # Save next layer

        visited.update(next_layer)  # Updated all visited layers (already discovered and assigned a layer)

        current_layer = next_layer  # Current layer is the next layer
        depth = depth + 1  # Increment depth

    return layers  # return list of layers: layers[d]


In [121]:
"""
# For each node in layer 1(d) count all edges between layer 1(d) and layer 0(d-1) --> (where d represents layer)
# For each node in layer 1(d), create two variables called pos_votes and neg_votes. 

# deletes edges based on previous layer
def filter_level(G, curr_layer, prev_layer, first,d):
    temp = 0 # DELETE LATER-----------------------
    for v in list(curr_layer):  # for every node v in the current layer
        
        pos_votes = 0
        neg_votes = 0
        pos_edges = []   # edges that voted positive
        neg_edges = []   # edges that voted negative

        for u in list(G.neighbors(v)):  # u is neighbors of v from previous layer
            if u in prev_layer:
                #if "direction" not in u and first==False:
                    #print("We will run into an error at",u)
                node_dir = float(G.nodes[u]["direction"])
                edge_dir = float(G[u][v]["direction"])
                vote = node_dir * edge_dir

                if vote > 0:
                    pos_votes += 1
                    pos_edges.append((u, v))
                else:
                    neg_votes += 1
                    neg_edges.append((u, v))

        log_file.write(f"\nNODE- {v},\nLAYER-{d},\nPOS_VOTES- {pos_votes},\nNEG_VOTES- {neg_votes}\n")

        if (v=="L-Lysine"):
            #print("Second TIME _-----------------------")
            print("Positive votes:",pos_votes)
            print("Negative votes:",neg_votes)
            print("Neighbors",list(G.neighbors(v)))
            #print("Second TIME _-----------------------")
        # if no votes, skip (otherwise dividing by zero)
        total = pos_votes + neg_votes
        if total == 0:
            print("ERROR")
            continue

        # set direction of v based on majority vote
        if pos_votes > neg_votes:
            G.nodes[v]["direction"] = 1
            minority_edges = neg_edges
        elif pos_votes < neg_votes:
            G.nodes[v]["direction"] = -1
            minority_edges = pos_edges
        else:
            minority_edges = pos_edges + neg_edges

        # calculate puc_value --> minority / total
        G.nodes[v]["puc_value"] = len(minority_edges) / total

        if G.nodes[v]["puc_value"] > 0.2:
            # delete all edges between v and nodes in the previous layer
            if first==True: #if layer 1, remove edge
                
                if ("direction" in G.nodes[v]):
                    del G.nodes[v]["direction"]
                for u in list(G.neighbors(v)):
                    if u in prev_layer:
                        G.remove_edge(u, v)
                        log_file.write(f"{v}-{u} edge is removed\n")
                        #print(first)
                log_file.write(f"{list(G.neighbors(v))} are neighbors of {v}\n")
            else: #all other layers, remove node
                if G.has_node(v):
                    G.remove_node(v)
                    log_file.write(f"{v} node is removed\n")
        else:
            # delete only minority-vote edges
            for a, b in minority_edges:
                G.remove_edge(a, b)
                log_file.write(f"{a} {b} edge is removed MINORITY CASE\n")
        if (v=="Dopamine"):
            print("Second TIME _-----------------------")
        
    return G


# deletes edges in same level
def same_level_edges(G, curr_layer,d):
    curr_layer = set(curr_layer)  

    for u in list(curr_layer): # for u in currrent layer
        if not G.has_node(u):
            continue
            
        for v in list(G.neighbors(u)): # for v in current layer that is a neighbor of u
            if v in curr_layer and G.has_node(v):
                if "direction" not in G.nodes[u] or "direction" not in G.nodes[v]:
                    #print(u,"is given another chance")
                    continue
                    
                if (v=="L-Glutamine" or u=="L-Glutamine"):
                    print("Not being removed",u,G.nodes[u]["direction"])
                    print("Not being removed",v,G.nodes[v]["direction"])
                
                node_prod = float(G.nodes[u]["direction"]) * float(G.nodes[v]["direction"])
                edge_dir = float(G[u][v]["direction"])

                if node_prod != edge_dir:  # if product of node directions != edge direction, delete edge
                    if G.has_edge(u, v):  # avoid double-delete since undirected
                        log_file.write(f"SAME EDGE REMOVAL {v}-{u} edge is removed in layer{d}\n\n\n")
                        G.remove_edge(u, v)

    return G
"""

'\n# For each node in layer 1(d) count all edges between layer 1(d) and layer 0(d-1) --> (where d represents layer)\n# For each node in layer 1(d), create two variables called pos_votes and neg_votes. \n\n# deletes edges based on previous layer\ndef filter_level(G, curr_layer, prev_layer, first,d):\n    temp = 0 # DELETE LATER-----------------------\n    for v in list(curr_layer):  # for every node v in the current layer\n        \n        pos_votes = 0\n        neg_votes = 0\n        pos_edges = []   # edges that voted positive\n        neg_edges = []   # edges that voted negative\n\n        for u in list(G.neighbors(v)):  # u is neighbors of v from previous layer\n            if u in prev_layer:\n                #if "direction" not in u and first==False:\n                    #print("We will run into an error at",u)\n                node_dir = float(G.nodes[u]["direction"])\n                edge_dir = float(G[u][v]["direction"])\n                vote = node_dir * edge_dir\n\n         

In [202]:
# For each node in layer 1(d) count all edges between layer 1(d) and layer 0(d-1) --> (where d represents layer)
# For each node in layer 1(d), create two variables called pos_votes and neg_votes. 

# deletes edges based on previous layer
def filter_level(G, curr_layer, prev_layer, first,d):
    temp = 0 # DELETE LATER-----------------------
    for v in list(curr_layer):  # for every node v in the current layer
        #print(v)
        pos_votes = 0
        neg_votes = 0
        pos_edges = []   # edges that voted positive
        neg_edges = []   # edges that voted negative

        for u in list(G.neighbors(v)):  # u is neighbors of v from previous layer
            if u in prev_layer:
                #if (v=="Albizziin" or u=="Albizziin"):
                        #print("Albizziin found")
                #if "direction" not in u and first==False:
                    #print("We will run into an error at",u)
                node_dir = float(G.nodes[u]["direction"])
                edge_dir = float(G[u][v]["direction"])
                vote = node_dir * edge_dir

                if vote > 0:
                    pos_votes += 1
                    log_file.write(f"{u} votes + \n")
                    pos_edges.append((u, v))
                elif vote < 0:
                    neg_votes += 1
                    log_file.write(f"{u} votes - \n")
                    neg_edges.append((u, v))
                else:
                    print("ZEROooooooooooooooooooooooooooooooooo")

        log_file.write(f"NODE- {v},\nLAYER-{d},\nPOS_VOTES- {pos_votes},\nNEG_VOTES- {neg_votes}\n")
        log_file.write(f"Neighbors:,{u}\n")

        if (v=="Albizziin"):
            #print("Second TIME _-----------------------")
            print("Positive votes:",pos_votes)
            print("Negative votes:",neg_votes)
            print("Neighbors",list(G.neighbors(v)))
            #print("Second TIME _-----------------------")
        # if no votes, skip (otherwise dividing by zero)
        total = pos_votes + neg_votes
        if total == 0:
            print("ERROR")
            continue

        # set direction of v based on majority vote
        if pos_votes > neg_votes:
            G.nodes[v]["direction"] = 1
            minority_edges = neg_edges
        elif pos_votes < neg_votes:
            G.nodes[v]["direction"] = -1
            minority_edges = pos_edges
        else:
            minority_edges = pos_edges + neg_edges

        # calculate puc_value --> minority / total
        G.nodes[v]["puc_value"] = len(minority_edges) / total

        if G.nodes[v]["puc_value"] > 0.2:
            # delete all edges between v and nodes in the previous layer
            if first==True: #if layer 1, remove edge
                
                if ("direction" in G.nodes[v]):
                    del G.nodes[v]["direction"]
                for u in list(G.neighbors(v)):
                    if u in prev_layer:
                        G.remove_edge(u, v)
                        log_file.write(f"{v}-{u} edge is removed\n")
                        #print(first)
                log_file.write(f"{list(G.neighbors(v))} are neighbors of {v}\n")
            else: #all other layers, remove node
                if G.has_node(v):
                    G.remove_node(v)
                    log_file.write(f"{v} node is removed\n")
        else:
            # delete only minority-vote edges
            for a, b in minority_edges:
                G.remove_edge(a, b)
                log_file.write(f"{a} {b} edge is removed MINORITY CASE\n")
        #if (v=="Dopamine"):
            #print("Second TIME _-----------------------")
        log_file.write("\n\n")
    return G


# deletes edges in same level
def same_level_edges(G, curr_layer,d):
    curr_layer = set(curr_layer)  

    for u in list(curr_layer): # for u in currrent layer
        if not G.has_node(u):
            continue
            
        for v in list(G.neighbors(u)): # for v in current layer that is a neighbor of u
            if v in curr_layer and G.has_node(v):
                if "direction" not in G.nodes[u] or "direction" not in G.nodes[v]:
                    #print(u,"is given another chance")
                    continue
                    
                #if (v=="L-Glutamine" or u=="L-Glutamine"):
                #    print("Not being removed",u,G.nodes[u]["direction"])
                #    print("Not being removed",v,G.nodes[v]["direction"])
                
                node_prod = float(G.nodes[u]["direction"]) * float(G.nodes[v]["direction"])
                edge_dir = float(G[u][v]["direction"])

                if node_prod != edge_dir:  # if product of node directions != edge direction, delete edge
                    if G.has_edge(u, v):  # avoid double-delete since undirected
                        log_file.write(f"SAME EDGE REMOVAL {v}-{u} edge is removed in layer{d}\n\n\n")
                        G.remove_edge(u, v)

    return G


In [223]:
import pandas as pd

log_file = open("log_file.txt","w")

#puc_delete_threshold=0.2
graph_A, layer_0 = build_network_by_index(
    "1. Filtered Files/cpx_nodes.csv",
    "1. Filtered Files/pls-pls.csv",
    "1. Filtered Files/pls-cpx.csv",
    has_header=True
)

if "Albizziin" in graph_A:
    print("Albizziin:")
    print("Neighbors:", list(graph_A.neighbors("Albizziin")),"\n")

if "3beta-TRIMETHYLSILOXY-5alpha,6alpha-EPOXYCHOLESTANE" in graph_A:
    print("3beta-TRIMETHYLSILOXY-5alpha,6alpha-EPOXYCHOLESTANE:")
    if ("Albizziin" in list(graph_A.neighbors("3beta-TRIMETHYLSILOXY-5alpha,6alpha-EPOXYCHOLESTANE"))):
        print("Neighbors: Albizziin")


    
layers = build_layers(graph_A, layer_0, max_depth=None)
print(graph_A)

d = 1 #Start from distance = 1
while d < len(layers):
    
    # DEBUG ------------------------------------------------------------------------------------------------------------
    #print("Total layers:", len(layers))
    # Number of nodes per layer
    sum = 0
    for i, layer in enumerate(layers):
        if i>0:
            sum = sum + len(layer)
            #print(f"Layer {i}: {len(layer)} nodes")
            #if (i>0):
                #print(f"Layer {i}: {layer} nodes")
    #print("Total number of nodes kept",sum)
    # ------------------------------------------------------------------------------------------------------------

    #print("Starting with:",d)
    prev_layer = set(layers[d - 1])   # define previous layer
    curr_layer = layers[d]   # define current layer
    #if ("Albizziin" in layers[d]):
    #    print("Albizziin found un til level",d)
    
    graph_A = filter_level(graph_A, curr_layer, prev_layer, False,d)
    graph_A = same_level_edges(graph_A, curr_layer,d)

        
    layers = build_layers(graph_A, layer_0, max_depth=None)
    #print("done with layer:",d,"\n")
    d += 1



rows = []
for d in range(1, len(layers)):
    for node in layers[d]:
        rows.append({
            "node": node,
            "direction": graph_A.nodes[node].get("direction"),
            "puc_value": graph_A.nodes[node].get("puc_value")
        })
    


print(len(rows))
df = pd.DataFrame(rows, columns=["node", "direction", "puc_value"])
df.to_csv("new.csv", index=False)


Albizziin:
Neighbors: ['2-Hydroxy-2-methylbutyric acid', '3beta-TRIMETHYLSILOXY-5alpha,6alpha-EPOXYCHOLESTANE', '5,6-Dimethyl-4-phenyl-3-cyanopyridine-2(1H)-thione'] 

3beta-TRIMETHYLSILOXY-5alpha,6alpha-EPOXYCHOLESTANE:
Neighbors: Albizziin
Graph with 1329 nodes and 4233 edges
374


In [231]:
layered = set().union(*layers)
not_in_layers = sorted(set(graph_A.nodes) - layered)
print("Nodes not in any layer:", not_in_layers)
print("Count:", len(not_in_layers))


Nodes not in any layer: ['1H-Inden-1-one, 2,3-dihydro-7-hydroxy-3-methyl- ', '1H-Indole, 1-(trimethysilyl)-2,5-bis[(trimethylsilyl)oxy]-', '2-Hydroxy-2-methylbutyric acid', '3-Methoxyamphetamine, 4-Trimethylsilyloxy, N,N-bis(Trimethylsilyloxy)-', 'Albizziin', 'Benzenepropanoic acid, 3,5-bis(1,1-dimethylethyl)-4-hydroxy-', 'ENSMUSG00000000127', 'ENSMUSG00000000305', 'ENSMUSG00000001014', 'ENSMUSG00000001089', 'ENSMUSG00000001441', 'ENSMUSG00000001855', 'ENSMUSG00000002319', 'ENSMUSG00000002455', 'ENSMUSG00000002748', 'ENSMUSG00000002844', 'ENSMUSG00000003099', 'ENSMUSG00000003184', 'ENSMUSG00000003199', 'ENSMUSG00000003344', 'ENSMUSG00000003360', 'ENSMUSG00000003527', 'ENSMUSG00000004085', 'ENSMUSG00000004591', 'ENSMUSG00000004698', 'ENSMUSG00000005045', 'ENSMUSG00000005124', 'ENSMUSG00000005131', 'ENSMUSG00000005469', 'ENSMUSG00000005575', 'ENSMUSG00000005609', 'ENSMUSG00000006281', 'ENSMUSG00000006800', 'ENSMUSG00000007379', 'ENSMUSG00000007476', 'ENSMUSG00000007570', 'ENSMUSG00000007

In [208]:
"""
create a graph G

Fec-Fec Edge --> 
Feci-PLS Edge
PLS-PLS Edge
PLS-CPX
CPX-CPX
CPX-CTX
CTX-CTX


"""

'\ncreate a graph G\n\nFec-Fec Edge --> \nFeci-PLS Edge\nPLS-PLS Edge\nPLS-CPX\nCPX-CPX\nCPX-CTX\nCTX-CTX\n\n\n'

In [241]:
import pandas as pd
import networkx as nx

# node_to_layer = final layers after filtering
node_to_layer = {}
for li, layer_nodes in enumerate(layers):
    for n in layer_nodes:
        node_to_layer[n] = li

# Find all nodes reachable from layer_0
reachable_from_layer0 = set()
for src in layer_0:
    if src in graph_A:
        reachable_from_layer0.update(nx.node_connected_component(graph_A, src))

# Edge pairs starting from layer 1 (same layer or adjacent layers)
edge_rows = []
for u, v in graph_A.edges():
    lu = node_to_layer.get(u)
    lv = node_to_layer.get(v)

    # Skip edges if either node is not reachable from layer_0
    if u not in reachable_from_layer0 or v not in reachable_from_layer0:
        continue

    # Skip edges involving layer_0 nodes themselves
    if lu == 0 or lv == 0:
        continue

    # Keep only same-layer or adjacent-layer edges
    if abs(lu - lv) > 1:
        continue
        
    edge_rows.append({"node1": u, "node2": v})

df_edges = pd.DataFrame(edge_rows, columns=["node1", "node2"])
df_edges.to_csv("new_pls-pls_node_pairs.csv", index=False)
print("done")
print("edges written:", len(edge_rows))


done
edges written: 1626


In [243]:
import pandas as pd

# node_to_layer = final layers after filtering
node_to_layer = {}
for li, layer_nodes in enumerate(layers):
    for n in layer_nodes:
        node_to_layer[n] = li

# Edge pairs starting from layer 1 (same layer or adjacent layers)
edge_rows = []
for u, v in graph_A.edges():
    lu = node_to_layer.get(u)
    lv = node_to_layer.get(v)

    # Start from layer 1 only (exclude minus layer 0)
    #if not (lu == 0 and lv == 1):

    if not ((lu == 0 and lv == 1) or (lu == 0 and lv == 0) or (lu == 1 and lv == 0)):
        continue
        
    edge_rows.append({"node1": u,"node2": v})

df_edges = pd.DataFrame(edge_rows, columns=["node1", "node2"])
df_edges.to_csv("new_cpx-pls_node_pairs.csv", index=False)
print("done")
print("edges written:", len(edge_rows))



done
edges written: 668
